In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from datasets import load_dataset, Audio

#Φόρτωση του dataset, από το HuggingFace
dataset = load_dataset("AbstractTTS/CREMA-D", split="train")

#Μετατροπή συχνότητας σε 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

print(dataset.features)
print(dataset[0]["file"], dataset[0]["major_emotion"])

### Φόρτωση συνόλου δεδομένων και speaker independent διαχωρισμός

In [ ]:
labels = sorted(set(dataset["major_emotion"]))   # ['anger','disgust','fear','happy','neutral','sad']
num_labels = len(labels)
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}

def add_speaker(batch):
    batch["actor_id"] = batch["file"].split("_")[0]   # π.χ. "1001_DFA_ANG_XX.wav" -> "1001"
    return batch

dataset = dataset.map(add_speaker, load_from_cache_file=False)
print(len(set(dataset["actor_id"])), "μοναδικοί ομιλητές")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

actor_ids = dataset["actor_id"]
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=actor_ids))

from datasets import DatasetDict
dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx)
})

# Έλεγχος ότι δεν υπάρχει επικάλυψη ομιλητών
train_speakers = set(dataset["train"]["actor_id"])
test_speakers = set(dataset["test"]["actor_id"])
print("Επικάλυψη:", train_speakers & test_speakers)  # πρέπει να είναι κενό σύνολο

### Φόρτωση του αρχικού προεκπαιδευμένου μοντέλου

In [ ]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
import torch

checkpoint = "facebook/hubert-base-ls960"
feature_extractor = AutoFeatureExtractor.from_pretrained(checkpoint)

model = AutoModelForAudioClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

model.freeze_feature_encoder()

if torch.cuda.is_available():
    model = model.to("cuda")

In [ ]:
MAX_LENGTH = 6 * 16000  # 6 δευτερόλεπτα στα 16kHz

#Συνάρτηση για την προετοιμασία του dataset
def prepare_dataset(batch):
    audio = batch["audio"]
    #Μετατροπή δείγματος ομιλίας σε κατάλληλη μορφή για το finetuning, με περικοπή στα 6 δευτερόλεπτα
    batch["input_values"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        max_length=MAX_LENGTH,
        truncation=True,
    ).input_values[0]
    #Μετατροπή της ετικέτας από string σε integer id, για συμβατότητα με τον trainer
    batch["labels"] = label2id[batch["major_emotion"]]
    return batch

#Εφαρμογή της συνάρτησης σε όλο το dataset και αφαίρεση των υπόλοιπων στηλών
encoded_dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    load_from_cache_file=False,
)

#Έλεγχος ότι οι κανονικοποιημένες τιμές είναι καθαρές (χωρίς ακραίες τιμές)
max_abs_vals = [np.max(np.abs(x["input_values"])) for x in encoded_dataset["train"]]
print("Μέγιστη απόλυτη τιμή (train):", max(max_abs_vals))

### Ορισμός μετρικών αξιολόγησης

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,

    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,

    roc_auc_score,
    average_precision_score
)

from sklearn.preprocessing import label_binarize
from scipy.special import softmax

In [ ]:
import numpy as np

#Λίστα για την αποθήκευση των confusion matrices
all_confusion_matrices = []

"""
Συνάρτηση για τον υπολογισμό των μετρικών ανά εποχή εκπαίδευσης, η οποία προσθέτει
τον confusion matrix κάθε εποχής στην all_confusion_matrices. Επειδή γίνεται εκ νέου
υπολογισμός της καλύτερης εποχής μετά το τέλος της εκπαίδευσης, αυτή περιέχει ένα
επιπλέον confusion matrix, ίδιο με το καλύτερο
"""

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)
    labels_bin = label_binarize(labels_true, classes=list(range(num_labels)))

    # ROC-AUC (macro-averaged, one-vs-rest)
    try:
        roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")

    # PR-AUC (macro-averaged average precision)
    try:
        pr_auc = average_precision_score(labels_bin, probs, average="macro")
    except ValueError:
        pr_auc = float("nan")

    #Σταθερό σχήμα πίνακα ακόμα κι αν κάποια κλάση δεν προβλεφθεί καθόλου
    cm = confusion_matrix(labels_true, predictions, labels=list(range(num_labels)))
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy_score(labels_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels_true, predictions),
        "precision": precision_score(labels_true, predictions, average="weighted", zero_division=0),
        "recall": recall_score(labels_true, predictions, average="weighted", zero_division=0),
        "f1": f1_score(labels_true, predictions, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(labels_true, predictions),
        "k_score": cohen_kappa_score(labels_true, predictions),
        "log_loss": log_loss(labels_true, probs, labels=list(range(num_labels))),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

### Ορισμός παραμέτρων εκπαίδευσης

In [ ]:
#Δυναμικός υπολογισμός warmup_steps (10% των συνολικών training steps)
per_device_train_batch_size = 8
gradient_accumulation_steps = 4
num_train_epochs = 8

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = len(encoded_dataset["train"]) // effective_batch_size
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = max(1, int(0.1 * total_steps))

#Ορισμός παραμέτρων εκπαίδευσης
training_args = TrainingArguments(
    output_dir="./hubert-finetuned-CREMA-D",
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=num_train_epochs,
    fp16=False,   # αποφυγή αστάθειας μικτής ακρίβειας στην οικογένεια wav2vec2
    learning_rate=3e-5,
    warmup_steps=warmup_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none",
)

#Data Collator: δυναμικό padding των input_values ανά batch
class DataCollatorCTCWithPadding:
    def __init__(self, feature_extractor, padding=True):
        self.feature_extractor = feature_extractor
        self.padding = padding

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        #Οι ετικέτες είναι ήδη integer ids από το prepare_dataset
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

In [ ]:
#Καθαρισμός των confusion matrix, από τυχόν προηγούμενες εκτελέσεις
all_confusion_matrices.clear()

### Δημιουργία και Εκπαίδευση του Trainer

In [ ]:
#Κλήση του trainer και εισαγωγή των παραμέτρων εκπαίδευσης
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Εκκίνηση εκπαίδευσης...")
trainer.train()
print("Εκπαίδευση ολοκληρώθηκε!")

In [ ]:
#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

In [ ]:
repo_id = "VasilisAsim/hubert-finetuned-CREMA-D"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")